## Introduction to deep learning - exercise

## Example

*Human Activity Recognition using Smartphones* dataset

Dataset description:

*The experiments have been carried out with a group of 30 volunteers. Each person performed six activities
(WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS, SITTING, STANDING, LAYING) wearing a smartphone.
Using its embedded accelerometer and gyroscope, we captured 3-axial linear acceleration and 3-axial angular velocity.
The experiments have been video-recorded to label the data manually.*

**Variables:**
For each record in the dataset it is provided:
* A 561-feature vector with time and frequency domain variables.
* Its activity label.
* An identifier of the subject who carried out the experiment.

More details at: https://archive.ics.uci.edu/ml/datasets/human+activity+recognition+using+smartphones

### Loading and preparing the data

In [1]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim

In [10]:
folder = '/Users/josepedro/Downloads/Uni/3ano/2semestre/Aprendizagem Computacional/datasets/'  ## put here folder where the file HAR_clean.csv is located

Load the dataset: "HAR_clean.csv"

In [11]:
all_data = pd.read_csv(os.path.join(folder, 'HAR_clean.csv'), index_col=0)

Divide into input and output data

In [12]:
input_data = all_data.iloc[:,:-2].values.astype(np.float32)
input_data.shape

(10299, 561)

In [13]:
output_data = all_data.iloc[:,-1].values
output_data.shape

(10299,)

Divide the data into train and test, keeping 30% for the test

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = input_data
y = output_data

# Split the data into training and testing sets
##########################################
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)
###########################################

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


### Best shallow ML model

In [24]:
from sklearn import svm
from sklearn.model_selection import GridSearchCV

parameters = {'kernel':['linear', 'rbf'], 'C':[1, 10, 100,1000], 'gamma':[0.01, 0.001]}

svm_model_d = svm.SVC()
opt_model_d = GridSearchCV(svm_model_d, parameters)

opt_model_d.fit(X_train, y_train)
print (opt_model_d.best_estimator_)


KeyboardInterrupt: 

In [23]:
opt_model_d.score(X_test, y_test)

NotFittedError: This GridSearchCV instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

### Training a DL model

In [25]:
from sklearn import preprocessing
le = preprocessing.LabelEncoder()
# LabelEncoder() is needed for DL models, as they require numerical labels
# sklearn does that internally for SVM, but we need to do it explicitly for DL models

le.fit(y_train)

y_train_encoded = le.transform(y_train)
y_test_encoded = le.transform(y_test)

In [26]:
X_train = torch.tensor(X_train)
y_train = torch.tensor(y_train_encoded)

X_test = torch.tensor(X_test)
y_test = torch.tensor(y_test_encoded)

In [58]:
# DNN model - define here the model - complete

class HARNet(nn.Module):
    def __init__(self, in_features, n_classes):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, n_classes)  # ← mudança aqui
        )

    def forward(self, x):
        return self.net(x)


n_features = X_train.shape[1]
n_classes = len(np.unique(y_train))

model = HARNet(n_features, n_classes)

In [59]:
# Loss & optimizer - complete !
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [60]:
from sklearn.metrics import accuracy_score

# Training loop
epochs = 30
batch_size = 128
train_losses = []
test_accuracies = []

for epoch in range(epochs):

    model.eval()
    # ---- mini-batch training ----
    perm = torch.randperm(X_train.size(0))
    epoch_loss = 0

    for i in range(0, X_train.size(0), batch_size):
        idx = perm[i:i + batch_size]
        xb, yb = X_train[idx], y_train[idx]

        preds = model(xb)
        loss = criterion(preds, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    train_losses.append(epoch_loss)

    model.eval()
    # ---- evaluation ----
    with torch.no_grad():
        logits = model(X_test)
        preds = torch.argmax(logits, dim=1)
        acc = accuracy_score(y_test.numpy(), preds.numpy())
        test_accuracies.append(acc)

    print(f"Epoch {epoch+1:02d} | Loss: {epoch_loss:.3f} | Test Acc: {acc:.4f}")

Epoch 01 | Loss: 45.568 | Test Acc: 0.8812
Epoch 02 | Loss: 13.088 | Test Acc: 0.9489
Epoch 03 | Loss: 7.747 | Test Acc: 0.9528
Epoch 04 | Loss: 5.576 | Test Acc: 0.9638
Epoch 05 | Loss: 4.386 | Test Acc: 0.9676
Epoch 06 | Loss: 3.808 | Test Acc: 0.9673
Epoch 07 | Loss: 3.632 | Test Acc: 0.9693
Epoch 08 | Loss: 3.157 | Test Acc: 0.9667
Epoch 09 | Loss: 2.840 | Test Acc: 0.9644
Epoch 10 | Loss: 2.812 | Test Acc: 0.9683
Epoch 11 | Loss: 2.517 | Test Acc: 0.9718
Epoch 12 | Loss: 2.625 | Test Acc: 0.9725
Epoch 13 | Loss: 2.287 | Test Acc: 0.9725
Epoch 14 | Loss: 1.971 | Test Acc: 0.9744
Epoch 15 | Loss: 2.167 | Test Acc: 0.9754
Epoch 16 | Loss: 1.896 | Test Acc: 0.9715
Epoch 17 | Loss: 1.991 | Test Acc: 0.9754
Epoch 18 | Loss: 1.639 | Test Acc: 0.9718
Epoch 19 | Loss: 1.632 | Test Acc: 0.9748
Epoch 20 | Loss: 1.383 | Test Acc: 0.9773
Epoch 21 | Loss: 1.380 | Test Acc: 0.9751
Epoch 22 | Loss: 1.261 | Test Acc: 0.9773
Epoch 23 | Loss: 1.202 | Test Acc: 0.9770
Epoch 24 | Loss: 1.137 | Test Ac

In [51]:
# Test set evaluation
print("\nFinal test accuracy:", test_accuracies[-1])


Final test accuracy: 0.9802588996763754
